In [17]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn as nn
import json

In [9]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [10]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = datasets.ImageFolder(
    root="data/train",
    transform=evaluation_transform,
)

evaluation_dataset = datasets.ImageFolder(
    root="data/test",
    transform=evaluation_transform,
)

In [11]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [13]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [15]:
model = HairModel()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

In [16]:
"""
What is the total number of trainable parameters? Count biases as well and use sum(p.numel() for p in model.parameters()).

    896
    11214912
    15896912
    20073473
"""
sum(p.numel() for p in model.parameters())

20073473

In [18]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": [],
}

for epoch in range(10):
    model.train()
    train_loss_sum = train_correct = train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        n = images.size(0)
        train_loss_sum += loss.item() * n
        train_correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
        train_total += n

    history["train_loss"].append(train_loss_sum / train_total)
    history["train_accuracy"].append(train_correct / train_total)

    model.eval()
    eval_loss_sum = eval_correct = eval_total = 0

    with torch.no_grad():
        for images, labels in evaluation_loader:
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)

            logits = model(images)
            loss = criterion(logits, labels)

            n = images.size(0)
            eval_loss_sum += loss.item() * n
            eval_correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
            eval_total += n

    history["evaluation_loss"].append(eval_loss_sum / eval_total)
    history["evaluation_accuracy"].append(eval_correct / eval_total)

    print(f"Epoch {epoch + 1}/10: train_loss={history['train_loss'][-1]:.4f}, "
          f"train_accuracy={history['train_accuracy'][-1]:.4f}, "
          f"evaluation_loss={history['evaluation_loss'][-1]:.4f}, "
          f"evaluation_accuracy={history['evaluation_accuracy'][-1]:.4f}")

with open("history_baseline.json", "w") as f:
    json.dump(history, f, indent=4)

Epoch 1/10: train_loss=0.6385, train_accuracy=0.6450, evaluation_loss=0.6010, evaluation_accuracy=0.6567
Epoch 2/10: train_loss=0.5441, train_accuracy=0.7175, evaluation_loss=0.6003, evaluation_accuracy=0.6517
Epoch 3/10: train_loss=0.4955, train_accuracy=0.7550, evaluation_loss=0.6405, evaluation_accuracy=0.6219
Epoch 4/10: train_loss=0.4330, train_accuracy=0.7913, evaluation_loss=0.6274, evaluation_accuracy=0.6915
Epoch 5/10: train_loss=0.4048, train_accuracy=0.8087, evaluation_loss=0.6293, evaluation_accuracy=0.6617
Epoch 6/10: train_loss=0.3953, train_accuracy=0.8075, evaluation_loss=0.6332, evaluation_accuracy=0.7114
Epoch 7/10: train_loss=0.3437, train_accuracy=0.8363, evaluation_loss=0.6895, evaluation_accuracy=0.6716
Epoch 8/10: train_loss=0.2806, train_accuracy=0.8850, evaluation_loss=0.6497, evaluation_accuracy=0.7214
Epoch 9/10: train_loss=0.2236, train_accuracy=0.9100, evaluation_loss=0.7114, evaluation_accuracy=0.7164
Epoch 10/10: train_loss=0.1584, train_accuracy=0.9387, 

In [19]:
"""
Calculate the median of train_accuracy in history_baseline.json. Submit the value rounded to two decimal places. The grader accepts an absolute error of 0.05; the answer is not selected from a fixed option list.
"""
with open("history_baseline.json") as f:
    history = json.load(f)

print(round(np.median(history["train_accuracy"]), 2))


0.81


In [20]:
"""
Calculate the population standard deviation of train_loss in history_baseline.json (for NumPy, use np.std(values, ddof=0)). Submit the value rounded to three decimal places. The grader accepts an absolute error of 0.02.
"""
with open("history_baseline.json") as f:
    history = json.load(f)

print(round(np.std(history["train_loss"], ddof=0), 3))


0.139


In [21]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset.transform = train_transform_augmented

history_augmented = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": [],
}

for epoch in range(10):
    model.train()
    train_loss_sum = train_correct = train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        n = images.size(0)
        train_loss_sum += loss.item() * n
        train_correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
        train_total += n

    history_augmented["train_loss"].append(train_loss_sum / train_total)
    history_augmented["train_accuracy"].append(train_correct / train_total)

    model.eval()
    eval_loss_sum = eval_correct = eval_total = 0

    with torch.no_grad():
        for images, labels in evaluation_loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            logits = model(images)
            loss = criterion(logits, labels)

            n = images.size(0)
            eval_loss_sum += loss.item() * n
            eval_correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
            eval_total += n

    history_augmented["evaluation_loss"].append(eval_loss_sum / eval_total)
    history_augmented["evaluation_accuracy"].append(eval_correct / eval_total)

    print(
        f"Epoch {epoch + 1}/10 | "
        f"Train Loss: {history_augmented['train_loss'][-1]:.4f} | "
        f"Train Accuracy: {history_augmented['train_accuracy'][-1]:.4f} | "
        f"Evaluation Loss: {history_augmented['evaluation_loss'][-1]:.4f} | "
        f"Evaluation Accuracy: {history_augmented['evaluation_accuracy'][-1]:.4f}"
    )

with open("history_augmented.json", "w") as f:
    json.dump(history_augmented, f, indent=4)

Epoch 1/10 | Train Loss: 0.6444 | Train Accuracy: 0.6400 | Evaluation Loss: 0.6283 | Evaluation Accuracy: 0.6866
Epoch 2/10 | Train Loss: 0.5865 | Train Accuracy: 0.6787 | Evaluation Loss: 0.7048 | Evaluation Accuracy: 0.6766
Epoch 3/10 | Train Loss: 0.5940 | Train Accuracy: 0.6613 | Evaluation Loss: 0.5812 | Evaluation Accuracy: 0.7114
Epoch 4/10 | Train Loss: 0.5469 | Train Accuracy: 0.7150 | Evaluation Loss: 0.6194 | Evaluation Accuracy: 0.6816
Epoch 5/10 | Train Loss: 0.5469 | Train Accuracy: 0.7113 | Evaluation Loss: 0.6176 | Evaluation Accuracy: 0.7065
Epoch 6/10 | Train Loss: 0.5207 | Train Accuracy: 0.7275 | Evaluation Loss: 0.5709 | Evaluation Accuracy: 0.7065
Epoch 7/10 | Train Loss: 0.5528 | Train Accuracy: 0.7037 | Evaluation Loss: 0.5715 | Evaluation Accuracy: 0.7114
Epoch 8/10 | Train Loss: 0.5313 | Train Accuracy: 0.7250 | Evaluation Loss: 0.6018 | Evaluation Accuracy: 0.6965
Epoch 9/10 | Train Loss: 0.5075 | Train Accuracy: 0.7450 | Evaluation Loss: 0.6582 | Evaluation 

In [22]:
"""
Calculate the mean of evaluation_loss in history_augmented.json. Submit the value rounded to three decimal places. The grader accepts an absolute error of 0.02.
"""
with open("history_augmented.json") as f:
    history = json.load(f)

print(round(np.mean(history["evaluation_loss"]), 3))

0.609


In [23]:
"""
Calculate the mean of the last five values in evaluation_accuracy in history_augmented.json (epochs 6 through 10 of the augmented phase). Submit the value rounded to two decimal places. The grader accepts an absolute error of 0.05.

The tolerances cover minor CPU wheel differences, while the fixed seed, loader, split, architecture, and inference transform prevent the large uncontrolled variation from the 2025 exercise.
"""
with open("history_augmented.json") as f:
    history = json.load(f)

print(round(np.mean(history["evaluation_accuracy"][-5:]), 2))

0.7
